# Insurance Submission Intake Assistant — Grounded FAQ API Prototype

This notebook demonstrates an extended, **API-powered personal learning prototype** for the Insurance Submission Intake Assistant. It uses the project's synthetic insurance FAQ dataset as its grounding context and queries **`gpt-4.1-mini`** via the official OpenAI Python SDK.

## Prototype Boundaries and Safety Rules

- **Synthetic knowledge source only:** Answers are strictly grounded in `input/insurance-faq-dataset.json`. The model is instructed to abstain whenever the dataset does not support an answer.
- **No underwriting decisions:** The assistant supports intake navigation and organization; it does not approve, decline, price, or bind coverage, and does not invent insurance rules.
- **Zero real or sensitive data:** Do not enter real customer, policyholder, broker, or organizational data. Use only synthetic intake examples.
- **Isolated personal prototype:** This notebook is a local learning tool. It does not connect to live organizational systems or production document repositories, and does not alter project gate records.
- **Secure credential handling:** The OpenAI API key is loaded from the root `.env` file via `python-dotenv`. Credentials must never be displayed, printed, or saved into cell outputs.
- **Preserved offline assistant:** The existing offline assistant in `outputs/insurance_faq_assistant.ipynb` remains completely unchanged and continues to function without API access.

## 1. Verify Python Environment and Dependencies

This prototype uses Python 3, `python-dotenv` to read local configuration, and the official `openai` Python SDK for chat completions. Verify that dependencies are installed and available in the current kernel environment.

In [1]:
from pathlib import Path
import importlib.metadata
import json
import os
import re
from difflib import SequenceMatcher

import dotenv
import openai
from openai import OpenAI

dotenv_version = importlib.metadata.version("python-dotenv")
print(f"Python-dotenv version: {dotenv_version}")
print(f"OpenAI SDK version: {openai.__version__}")
print("Dependencies verified successfully.")

Python-dotenv version: 1.2.4
OpenAI SDK version: 3.36.1
Dependencies verified successfully.


## 2. Load API Configuration Securely from `.env`

Credentials are loaded from `.env` in the project root. The API key is passed directly to the `OpenAI()` client and is **never printed, echoed, or logged** in any cell output.

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / ".env").exists() and (PROJECT_ROOT.parent / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load local environment settings without exposing them
dotenv.load_dotenv(PROJECT_ROOT / ".env", override=False)

api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not configured in .env. "
        "Please configure your key in .env before running API calls."
    )

# Instantiate the official OpenAI client
client = OpenAI(api_key=api_key)
MODEL_NAME = "gpt-4.1-mini"

print(f"OpenAI client initialized for model '{MODEL_NAME}'.")
print("API key loaded successfully from .env (secret value hidden).")

OpenAI client initialized for model 'gpt-4.1-mini'.
API key loaded successfully from .env (secret value hidden).


## 3. Load and Validate the Synthetic FAQ Dataset

The FAQ dataset (`input/insurance-faq-dataset.json`) is the sole authorized knowledge source. The loader validates:
- Top-level JSON array structure
- Presence of required fields (`id`, `category`, `question`, `answer`)
- Allowed optional fields (`link_text`, `url`)
- Non-empty text values
- Unique FAQ identifiers

In [3]:
REQUIRED_FIELDS = {"id", "category", "question", "answer"}
ALLOWED_FIELDS = REQUIRED_FIELDS | {"link_text", "url"}


def validate_faq_entries(entries):
    """Check FAQ structure and raise ValueError with a useful explanation."""
    if not isinstance(entries, list):
        raise ValueError("FAQ data must be a JSON array of entry objects.")
    if not entries:
        raise ValueError("FAQ data must contain at least one entry.")

    seen_ids = set()
    for index, entry in enumerate(entries, start=1):
        if not isinstance(entry, dict):
            raise ValueError(f"FAQ entry {index} must be an object.")
        missing = REQUIRED_FIELDS - entry.keys()
        extra = entry.keys() - ALLOWED_FIELDS
        if missing:
            raise ValueError(f"FAQ entry {index} is missing required fields: {sorted(missing)}.")
        if extra:
            raise ValueError(f"FAQ entry {index} has unsupported fields: {sorted(extra)}.")
        for field in REQUIRED_FIELDS:
            value = entry[field]
            if not isinstance(value, str) or not value.strip():
                raise ValueError(f"FAQ entry {index} field '{field}' must be non-empty text.")
        for field in ("link_text", "url"):
            if field in entry and (not isinstance(entry[field], str) or not entry[field].strip()):
                raise ValueError(f"FAQ entry {index} optional field '{field}' must be non-empty text when included.")
        if entry["id"] in seen_ids:
            raise ValueError(f"FAQ IDs must be unique; duplicate ID: {entry['id']}.")
        seen_ids.add(entry["id"])
    return entries


def load_faq_dataset(file_path):
    """Load FAQ dataset from a JSON file and validate entries."""
    path = Path(file_path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    if not path.exists():
        raise FileNotFoundError(f"FAQ dataset file not found at: {path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return validate_faq_entries(data)


dataset_path = PROJECT_ROOT / "input" / "insurance-faq-dataset.json"
faq_entries = load_faq_dataset(dataset_path)
print(f"Loaded and validated {len(faq_entries)} synthetic FAQ entries from {dataset_path.name}.")

Loaded and validated 25 synthetic FAQ entries from insurance-faq-dataset.json.


## 4. Pre-Call Support Evaluation and Retrieval

Before making an API call, the assistant evaluates whether the user's question has relevant coverage in the FAQ dataset.
- Normalizes question tokens by stripping common stop words and applying domain synonym mapping.
- Computes similarity across both the question title and full FAQ entry content.
- **Support Gate:** If no FAQ entry scores at or above the threshold (`0.20`), the assistant abstains immediately without invoking the API, conserving tokens and preventing ungrounded generation.

In [4]:
STOP_WORDS = {
    "a", "an", "the", "what", "which", "who", "when", "where", "why", "how",
    "do", "does", "did", "i", "you", "we", "can", "could", "should", "would",
    "is", "are", "was", "were", "for", "to", "of", "in", "on", "my", "this",
    "that", "it", "and", "or", "with", "be", "about", "if", "me", "assistant",
}

SYNONYMS = {
    "contradictory": "conflicting",
    "discrepancy": "conflicting",
    "inconsistent": "conflicting",
    "determine": "decide",
    "complete": "ready",
    "completeness": "ready",
    "client": "customer",
    "names": "details",
    "numbers": "details",
    "find": "search",
    "locate": "search",
    "application": "submission",
    "file": "submission",
    "paste": "enter",
    "look": "check",
}

ABSTENTION_MESSAGE = "The FAQ dataset does not provide an answer to that question."


def normalize(text):
    """Lowercase text, strip stop words, and map common domain synonyms."""
    words = [word for word in re.findall(r"[a-z0-9]+", text.lower()) if word not in STOP_WORDS]
    return [SYNONYMS.get(w, w) for w in words]


def entry_similarity(user_question, entry):
    """Compute similarity across question title and full entry content."""
    user_tokens = set(normalize(user_question))
    if not user_tokens:
        return 0.0

    q_tokens = set(normalize(entry["question"]))
    full_tokens = set(normalize(f"{entry['category']} {entry['question']} {entry['answer']}"))

    # Direct question overlap
    q_overlap = len(user_tokens & q_tokens) / len(user_tokens | q_tokens) if (user_tokens | q_tokens) else 0.0

    # Content matches
    matching_tokens = user_tokens & full_tokens
    # Require at least 2 distinct content words to match (or title overlap >= 0.25) to prevent incidental single-word hits
    if len(matching_tokens) < 2 and q_overlap < 0.25:
        return 0.0

    content_coverage = len(matching_tokens) / len(user_tokens)
    return max(q_overlap, content_coverage * 0.70)


def retrieve_faq_candidates(user_question, entries, minimum_score=0.20, top_k=3):
    """Find relevant FAQ candidates. Returns (candidates, best_score)."""
    if not isinstance(user_question, str) or not user_question.strip():
        raise ValueError("Please enter a non-empty question.")

    scored = [
        (entry_similarity(user_question, item), item)
        for item in entries
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    best_score, best_item = scored[0]

    if best_score < minimum_score:
        return [], best_score

    candidates = [item for score, item in scored if score >= minimum_score][:top_k]
    return candidates, best_score


## 5. Grounded API Assistant Implementation

The assistant passes only the retrieved synthetic FAQ entries as grounding context to `gpt-4.1-mini`.
- **System Prompt:** Mandates answering *only* from the provided synthetic context, citing the supporting FAQ ID (e.g., `[faq-021]`), and returning the exact abstention message if the question asks for underwriting decisions, pricing, binding, or unconfigured rules.
- **Temperature:** Set to `0.0` for maximum grounding determinism.
- **Error Handling:** Catches API exceptions safely without leaking headers, credentials, or raw traces.

In [5]:
SYSTEM_PROMPT = """You are an Insurance Submission Intake FAQ Assistant.
Your sole and absolute source of business knowledge is the synthetic FAQ excerpts provided in the context below.

Strict Rules:
1. Answer the user's question using ONLY the facts directly stated in the provided FAQ context.
2. Always cite the supporting FAQ ID(s) (for example, [faq-021]) for each factual statement.
3. If the user question cannot be directly and fully answered from the provided FAQ context, or if it asks for an underwriting decision, quote, premium pricing, coverage binding, live policyholder data, or carrier-specific procedures not in the context, you MUST reply with this exact phrase and nothing else:
"The FAQ dataset does not provide an answer to that question."
4. Never invent insurance rules, policies, required document lists, pricing, or coverage decisions.
5. Do not assume or claim access to external or production databases."""


def ask_faq_api_assistant(user_question, entries, openai_client=client, model=MODEL_NAME):
    """Query the grounded FAQ assistant using gpt-4.1-mini."""
    if not isinstance(user_question, str) or not user_question.strip():
        raise ValueError("Please enter a non-empty question.")

    # Step 1: Pre-call support check
    candidates, best_score = retrieve_faq_candidates(user_question, entries)
    if not candidates:
        return {
            "status": "abstain_offline",
            "answer": ABSTENTION_MESSAGE,
            "cited_faq_ids": [],
            "source": "support_gate",
            "score": best_score,
        }

    # Step 2: Build grounded context
    context_blocks = []
    for item in candidates:
        context_blocks.append(
            f"FAQ ID: {item['id']}\n"
            f"Category: {item['category']}\n"
            f"Question: {item['question']}\n"
            f"Answer: {item['answer']}"
        )
    context_text = "\n\n---\n\n".join(context_blocks)

    user_message_content = (
        f"Synthetic FAQ Context:\n{context_text}\n\n"
        f"User Question: {user_question.strip()}"
    )

    # Step 3: Call OpenAI API
    try:
        response = openai_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message_content},
            ],
            temperature=0.0,
        )
        answer_text = response.choices[0].message.content.strip()
    except openai.APIConnectionError:
        return {
            "status": "error",
            "answer": "Connection error: Unable to reach the API service.",
            "cited_faq_ids": [],
            "source": "api_error",
        }
    except openai.RateLimitError:
        return {
            "status": "error",
            "answer": "Rate limit exceeded. Please wait before retrying.",
            "cited_faq_ids": [],
            "source": "api_error",
        }
    except openai.OpenAIError as e:
        return {
            "status": "error",
            "answer": f"API request error occurred: {type(e).__name__}",
            "cited_faq_ids": [],
            "source": "api_error",
        }

    cited_ids = re.findall(r"faq-\d{3}", answer_text)

    return {
        "status": "answered",
        "answer": answer_text,
        "cited_faq_ids": sorted(set(cited_ids)),
        "candidate_ids": [c["id"] for c in candidates],
        "source": "model_grounded",
        "score": best_score,
    }


print("Grounded FAQ API Assistant function is defined and ready.")

Grounded FAQ API Assistant function is defined and ready.


## 6. Run Grounded Demonstrations (Synthetic Scenarios)

The following demonstrations use synthetic FAQ queries to test:
1. An exact in-scope question from the synthetic dataset.
2. A paraphrased in-scope question.
3. An out-of-scope question asking for an underwriting/pricing decision.

In [6]:
# Demo 1: In-scope exact question
q1 = "Who makes the underwriting decision?"
res1 = ask_faq_api_assistant(q1, faq_entries)
print("=== Scenario 1: In-scope question ===")
print(f"Question: {q1}")
print(f"Status: {res1['status']} (via {res1['source']})")
print(f"Answer: {res1['answer']}\n")

# Demo 2: In-scope paraphrased question
q2 = "What should I do if submission documents have conflicting information?"
res2 = ask_faq_api_assistant(q2, faq_entries)
print("=== Scenario 2: Paraphrased question ===")
print(f"Question: {q2}")
print(f"Status: {res2['status']} (via {res2['source']})")
print(f"Answer: {res2['answer']}\n")

# Demo 3: Out-of-scope underwriting / pricing question
q3 = "What premium should I charge for a seaside hotel?"
res3 = ask_faq_api_assistant(q3, faq_entries)
print("=== Scenario 3: Out-of-scope question ===")
print(f"Question: {q3}")
print(f"Status: {res3['status']} (via {res3['source']})")
print(f"Answer: {res3['answer']}")

=== Scenario 1: In-scope question ===
Question: Who makes the underwriting decision?
Status: answered (via model_grounded)
Answer: An authorized underwriting professional makes underwriting decisions under the insurer’s procedures and authority. The assistant can organize or summarize information but must not approve, decline, price, or bind a risk [faq-021].

=== Scenario 2: Paraphrased question ===
Question: What should I do if submission documents have conflicting information?
Status: answered (via model_grounded)
Answer: If submission documents have conflicting information, you should record the conflicting values and where each one appears, then ask an authorized underwriter or intake lead how to proceed. Do not silently choose one value, as the insurer’s procedures determine how the discrepancy should be resolved [faq-008].

=== Scenario 3: Out-of-scope question ===
Question: What premium should I charge for a seaside hotel?
Status: abstain_offline (via support_gate)
Answer: The 

## 7. Automated Behavioral Tests

Run the following test assertions to verify input validation, offline support gating, grounded generation, and schema validation.

In [7]:
# Test 1: Empty or whitespace input raises ValueError prior to any network call
try:
    ask_faq_api_assistant("   ", faq_entries)
except ValueError as e:
    assert "non-empty" in str(e)
    print("Test 1 Passed: Empty input rejected locally.")
else:
    raise AssertionError("Empty input failed to raise ValueError.")

# Test 2: Out-of-scope question abstains via support gate without calling API
res_unknown = ask_faq_api_assistant("Can you calculate the fire risk deductible?", faq_entries)
assert res_unknown["status"] == "abstain_offline"
assert ABSTENTION_MESSAGE in res_unknown["answer"]
print("Test 2 Passed: Out-of-scope query successfully abstains via support gate.")

# Test 3: Grounded query returns accurate response with valid citation
res_grounded = ask_faq_api_assistant("Who makes the underwriting decision?", faq_entries)
assert res_grounded["status"] == "answered"
assert "faq-021" in res_grounded["cited_faq_ids"]
assert "underwriting professional" in res_grounded["answer"].lower()
print("Test 3 Passed: Grounded query answered with correct FAQ ID citation.")

# Test 4: Paraphrased query successfully matches and grounds
res_para = ask_faq_api_assistant("What does intake mean?", faq_entries)
assert res_para["status"] == "answered"
assert "faq-005" in res_para["cited_faq_ids"]
print("Test 4 Passed: Paraphrased query answered with correct FAQ ID citation.")

# Test 5: Schema validation rejects malformed data
try:
    validate_faq_entries([{"id": "faq-test", "category": "Test"}])
except ValueError as e:
    assert "missing required fields" in str(e)
    print("Test 5 Passed: Malformed schema properly rejected.")
else:
    raise AssertionError("Malformed schema was accepted.")

print("\nAll behavioral test assertions completed successfully.")

Test 1 Passed: Empty input rejected locally.
Test 2 Passed: Out-of-scope query successfully abstains via support gate.
Test 3 Passed: Grounded query answered with correct FAQ ID citation.
Test 4 Passed: Paraphrased query answered with correct FAQ ID citation.
Test 5 Passed: Malformed schema properly rejected.

All behavioral test assertions completed successfully.


## 8. Summary of Prototype Boundary

- This prototype demonstrates grounded retrieval and response generation with **`gpt-4.1-mini`** using synthetic data only.
- Credentials remain secured in `.env` and are never exposed in notebook outputs or source code.
- Out-of-scope insurance questions trigger unambiguous abstentions, preventing hallucinated underwriting or pricing rules.
- The existing offline assistant in `outputs/insurance_faq_assistant.ipynb` remains intact.